# Persona pilot: Qwen2.5-1.5B + Hinglish adapter (merged) + persona LoRA

The same persona pilot as `persona-pilot-gemma.ipynb` run 4, with one change: the decoder.

**Why this decoder.** In the Hinglish comparison (`compare-hinglish-2models*.ipynb`), `Qwen/Qwen2.5-1.5B` with the Hinglish adapter `sarthaksenapati/qwen1.5b-hinglish-sft-v2` wrote fluent Roman-script Hinglish, but never stopped: it hit the 80-token limit 5/5 in both passes and drifted. Gemma was chosen instead. Since then, every persona run has shown that LoRA teaches stopping, because the end-of-turn token is part of the trained target (Gemma: never stopping untouched → 84/84 in run 4). Gemma 3 4B's weak point is coherence; Qwen + adapter's strength was fluency. So: merge the Hinglish adapter into the base, train a persona LoRA on top, and see whether it now stops *and* writes better.

**Setup**

1. Load `Qwen/Qwen2.5-1.5B` in fp32 (6 GB, fits a T4; no quantization needed), apply the Hinglish adapter, **merge it into the weights** (`merge_and_unload`). The result is one plain model with Hinglish built in.
2. Train a **new persona LoRA** on top: r 16 / α 32 on attention + MLP, learning rate 1e-4, up to 8 epochs, early stopping after 2 epochs without a lower held-out loss, best held-out epoch kept.
3. The target of every training example ends with Qwen's `<|im_end|>`, so the loss teaches the line *and* where to stop.
4. Prompt format: the adapter's own chat template, user turn only, no system turn (how the adapter was trained).

**Same as Gemma run 4:** the 168 examples (120 train, 48 held out, disjoint scenarios), the persona trait lines, the six fixed new scenarios, micro-batch 2 × 2, greedy decoding with the 3-gram guard, probe outputs at every epoch, the evaluation and the blind sheet. "Untouched" here means Qwen + Hinglish merged, without the persona LoRA.

**Success bar** (unchanged from Gemma run 4, LoRA outputs on all 84 evaluation inputs):

| Check | Gemma 4B run 4 | Pass |
|---|---|---|
| Coherence: natural, on-topic lines (human) | about 1 in 3 | at least 2 in 3 |
| Persona recognition: blind matching, Cell 8 (human) | not done | at least half right |
| Distinctiveness: mean overlap between personas | 0.142 | ≤ 0.15 |
| Termination: ended on a stop token | 84/84 | ≥ 80/84 |
| Repetition: outputs repeating a sentence | 0/84 | ≤ 4/84 |
| Generalization | held on both | the checks hold on new scenarios too |

## Cell 1 — Setup and checks on the two repositories

No Hugging Face token is needed (both repositories are public). Each repository is pinned to the revision found now, and the adapter is checked against the base exactly as in the Hinglish comparison: its configured base model must be `Qwen/Qwen2.5-1.5B`, and the base's latest revision must predate the adapter's first commit. fp32 throughout; greedy decoding, 80 new tokens at most.

In [16]:
# Cell 1 — Setup: packages, settings, versions, pinned revisions, adapter check

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import json, random, re, time, urllib.request
from itertools import combinations

import peft
import torch
import transformers
from huggingface_hub import HfApi

BASE, ADAPTER = "Qwen/Qwen2.5-1.5B", "sarthaksenapati/qwen1.5b-hinglish-sft-v2"
DATA_URL = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
            "hindi-decoder-compare/project4-hindi-decoder/data/persona_pilot_168.json")
MAX_NEW_TOKENS = 80
SEED = 42
DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

api = HfApi()
REV = {r: api.model_info(r).sha for r in (BASE, ADAPTER)}
adapter_cfg = json.load(open(api.hf_hub_download(ADAPTER, "adapter_config.json", revision=REV[ADAPTER])))
base_latest = api.list_repo_commits(BASE)[0].created_at
adapter_first = api.list_repo_commits(ADAPTER)[-1].created_at
assert adapter_cfg["base_model_name_or_path"] == BASE, adapter_cfg["base_model_name_or_path"]
assert base_latest < adapter_first, "the base changed after the adapter was trained"

VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "cpu",
    "dtype": str(DTYPE),
    "revisions": REV,
    "hinglish adapter": {k: adapter_cfg.get(k) for k in ("r", "lora_alpha", "target_modules")},
}
print(json.dumps(VERSIONS, indent=1, default=str))

adapter_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

{
 "torch": "2.11.0+cu128",
 "transformers": "5.17.0",
 "peft": "0.21.2",
 "device": "Tesla T4",
 "dtype": "torch.float32",
 "revisions": {
  "Qwen/Qwen2.5-1.5B": "8faed761d45a263340a0528343f099c05c9a4323",
  "sarthaksenapati/qwen1.5b-hinglish-sft-v2": "942fd0b2ebd2119d772f18952de6f8d372a6448b"
 },
 "hinglish adapter": {
  "r": 32,
  "lora_alpha": 64,
  "target_modules": [
   "gate_proj",
   "v_proj",
   "k_proj",
   "o_proj",
   "q_proj",
   "down_proj",
   "up_proj"
  ]
 }
}


## Cell 2 — Data and the split

The 168 examples come from the repository, each tagged `train` or `eval`; training and evaluation scenarios do not overlap. The six fixed new scenarios (from runs 1–3) are added for evaluation only. Every input is built the same way, the persona's broad traits followed by the scenario and the instruction:

```
[Persona: Don] Traits: <two lines>
Scenario: ... Generate one original Roman-script Hinglish dialogue line ... Do not quote existing films.
```

In [17]:
# Cell 2 — Load the 168 examples (120 train, 48 held out), add the six fixed new scenarios

DATA = json.load(urllib.request.urlopen(DATA_URL))
PERSONAS = list(dict.fromkeys(ex["persona"] for ex in DATA))
SCENARIOS = list(dict.fromkeys(ex["scenario"] for ex in DATA))
TAIL = (" Generate one original Roman-script Hinglish dialogue line with this persona's broad dramatic traits."
        " Do not quote existing films.")

HELD_OUT = list(dict.fromkeys(ex["scenario"] for ex in DATA if ex["split"] == "eval"))
NEW = [
    "Boss ne Friday shaam ko achaanak weekend ka kaam de diya.",
    "Tumhara purana dost saalon baad airport par mil gaya.",
    "Presentation se paanch minute pehle office ka WiFi band ho gaya.",
    "Baarish mein gaadi beech sadak par band ho gayi.",
    "Team ka sabse senior saathi aaj retire ho raha hai.",
    "Kisi ne tumse kaha ki tum yeh kaam kabhi nahi kar paoge.",
]
assert not set(NEW) & set(SCENARIOS)

TRAITS = {
    "Vijay_Deewar": "Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, "
                    "speaks with quiet defiance and moral weight; never begs, never backs down.",
    "Don": "Don: a cool, calculating boss; confident, faintly amused, always a step ahead; "
           "short, controlled lines with a hint of threat.",
    "Shahenshah": "Shahenshah: a larger-than-life champion of justice with a royal manner; theatrical and lofty, "
                  "speaks in grand, Urdu-tinged words about honour, truth and dignity.",
    "Anthony_Gonsalves": "Anthony Gonsalves: a cheerful, comic Bombay boy; playful and warm, exaggerates for laughs, "
                         "mixes English freely, full of energy and jokes.",
    "Agneepath_Vijay": "Vijay Dinanath Chauhan (Agneepath): intense and terse; controlled anger, "
                       "short blunt sentences, driven by a long-held resolve.",
    "Jai_Sholay": "Jai (Sholay): calm, laconic, dry humour; a loyal, practical friend "
                  "who says little and keeps things steady.",
}
assert set(TRAITS) == set(PERSONAS)


def make_input(persona, scenario):
    return f"[Persona: {persona}] Traits: {TRAITS[persona]}\nScenario: {scenario}{TAIL}"


for ex in DATA:
    ex["input"] = make_input(ex["persona"], ex["scenario"])

TRAIN = [ex for ex in DATA if ex["split"] == "train"]
HELD = [ex for ex in DATA if ex["split"] == "eval"]
assert not {ex["scenario"] for ex in TRAIN} & set(HELD_OUT)
EVAL = [{"persona": p, "scenario": s, "kind": "held-out" if s in HELD_OUT else "new",
         "input": make_input(p, s), "reference": next((ex["target"] for ex in HELD
                                                       if ex["persona"] == p and ex["scenario"] == s), None)}
        for s in HELD_OUT + NEW for p in PERSONAS]

print(f"personas {PERSONAS}")
print(f"train {len(TRAIN)} examples ({len({ex['scenario'] for ex in TRAIN})} scenarios) | "
      f"held out {len(HELD)} ({len(HELD_OUT)} scenarios) | evaluation inputs {len(EVAL)} "
      f"({len(HELD_OUT)} held-out + {len(NEW)} new scenarios)")
print("example input:\n" + TRAIN[0]["input"])

personas ['Vijay_Deewar', 'Don', 'Shahenshah', 'Anthony_Gonsalves', 'Agneepath_Vijay', 'Jai_Sholay']
train 120 examples (20 scenarios) | held out 48 (8 scenarios) | evaluation inputs 84 (8 held-out + 6 new scenarios)
example input:
[Persona: Vijay_Deewar] Traits: Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, speaks with quiet defiance and moral weight; never begs, never backs down.
Scenario: Abhee chai ka mood hai. Generate one original Roman-script Hinglish dialogue line with this persona's broad dramatic traits. Do not quote existing films.


## Cell 3 — Merge the Hinglish adapter, set up the template, the training tensors and `generate`

- The base is loaded in fp32, the Hinglish adapter applied with `PeftModel.from_pretrained` and merged into the weights with `merge_and_unload`. From here on it is one ordinary model.
- The tokenizer and chat template come from the adapter repository (plain ChatML, user turn only). Stop tokens: `<|im_end|>` and `<|endoftext|>`.
- Each example is one user turn and one assistant turn. The part after the prompt (the line plus the template's closing `<|im_end|>`) is taken from the template's own rendering, so the closing tokens are exactly what the template writes. Labels are `-100` on the prompt, so the loss covers the line and the stop.
- `generate`: greedy, 80 new tokens at most, no repeated word 3-grams (`no_repeat_ngram_size=3`), as in Gemma run 4.

In [18]:
# Cell 3 — Load Qwen, merge the Hinglish adapter, render examples with its template, mask the prompt out

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ADAPTER, revision=REV[ADAPTER])
base = AutoModelForCausalLM.from_pretrained(BASE, revision=REV[BASE], dtype=DTYPE).to(DEVICE)
base = PeftModel.from_pretrained(base, ADAPTER, revision=REV[ADAPTER]).merge_and_unload()
assert not any("lora_" in n for n, _ in base.named_parameters()), "the Hinglish adapter is not merged"
print(f"GPU memory after loading and merging: {torch.cuda.memory_allocated(DEVICE) / 2**30:.1f} GB")
STOPS = sorted({tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")})
PAD = tok.pad_token_id if tok.pad_token_id is not None else STOPS[0]


def prompt_text(user):
    msgs = [{"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)


def encode(ex):
    """Prompt tokens + answer tokens, tokenized apart (exactly what generation sees). The answer is the template's
    own rendering after the prompt: the line plus its closing <|im_end|>."""
    msgs = [{"role": "user", "content": ex["input"]}, {"role": "assistant", "content": ex["target"]}]
    full, head = tok.apply_chat_template(msgs, tokenize=False), prompt_text(ex["input"])
    assert full.startswith(head), "the generation prompt is not a prefix of the full conversation"
    answer = full[len(head):]
    assert answer.startswith(ex["target"]) and "<|im_end|>" in answer, repr(answer)
    prompt = tok(head, add_special_tokens=False)["input_ids"]
    reply = tok(answer, add_special_tokens=False)["input_ids"]
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def collate(examples):
    pairs = [encode(ex) for ex in examples]
    T = max(len(ids) for ids, _ in pairs)
    ids = torch.full((len(pairs), T), PAD)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": mask.to(DEVICE), "labels": labels.to(DEVICE)}


DEVA = re.compile(r"[ऀ-ॿ]")
LATN = re.compile(r"[A-Za-z]")
NO_REPEAT = 3


@torch.no_grad()
def generate(user):
    ids = tok(prompt_text(user), return_tensors="pt", add_special_tokens=False).to(DEVICE)
    out = model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS,
                         no_repeat_ngram_size=NO_REPEAT, pad_token_id=PAD,
                         temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    text = tok.decode(new, skip_special_tokens=True).strip()
    d, l = len(DEVA.findall(text)), len(LATN.findall(text))
    return {"output": text, "new_tokens": len(new),
            "ended": "stop token" if new and new[-1] in STOPS else "token limit",
            "devanagari_share": round(d / (d + l), 3) if d + l else None}


ids, labels = encode(TRAIN[0])
print("stop token IDs", STOPS, tok.convert_ids_to_tokens(STOPS))
print("--- one training example as the model sees it:")
print(tok.decode(ids))
print("--- the part that carries the loss:")
print(repr(tok.decode(ids[labels != -100])))

tokenizer_config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/180 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  148MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

GPU memory after loading and merging: 10.7 GB
stop token IDs [151643, 151645] ['<|endoftext|>', '<|im_end|>']
--- one training example as the model sees it:
<|im_start|>user
[Persona: Vijay_Deewar] Traits: Vijay (Deewar): a self-made man who grew up poor; proud, bitter about injustice, speaks with quiet defiance and moral weight; never begs, never backs down.
Scenario: Abhee chai ka mood hai. Generate one original Roman-script Hinglish dialogue line with this persona's broad dramatic traits. Do not quote existing films.<|im_end|>
<|im_start|>assistant
Chai peene ka mann hai, toh chai piyenge. Har chhoti khushi ke liye kisi ki ijaazat nahi maangte hum.<|im_end|>

--- the part that carries the loss:
'Chai peene ka mann hai, toh chai piyenge. Har chhoti khushi ke liye kisi ki ijaazat nahi maangte hum.<|im_end|>\n'


## Cell 4 — Persona LoRA training with early stopping and probe outputs

- A new LoRA (rank 16, alpha 32, dropout 0.05) on the merged model's `q/k/v/o` and `gate/up/down` projections; everything else frozen, all in fp32.
- Micro-batches of 2 with 2-step gradient accumulation (effective batch 4); AdamW, learning rate 1e-4, linear decay over the 8-epoch maximum.
- After every epoch: training loss, loss on the 48 held-out examples, and the **probe** (one held-out and one new scenario, all six personas).
- Early stopping after two epochs without a lower held-out loss; the best held-out epoch is kept. Epoch 0 is Qwen + Hinglish without the persona LoRA.

In [19]:
# Cell 4 — Wrap with LoRA, train with early stopping, print probe outputs after every epoch

from peft import LoraConfig, get_peft_model

MAX_EPOCHS, PATIENCE, MICRO, ACCUM, LR = 8, 2, 2, 2, 1e-4
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
PROBE = [HELD_OUT[0], NEW[0]]

torch.manual_seed(SEED)
model = get_peft_model(base, LORA)
model.print_trainable_parameters()

trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / steps)


@torch.no_grad()
def held_out_loss(batch=4):
    model.eval()
    total = 0.0
    for i in range(0, len(HELD), batch):
        chunk = HELD[i:i + batch]
        total += model(**collate(chunk)).loss.item() * len(chunk)
    return total / len(HELD)


def lora_state():
    return {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}


def probe(epoch):
    model.eval()
    rows = [{"epoch": epoch, "scenario": s, "persona": p, **generate(make_input(p, s))}
            for s in PROBE for p in PERSONAS]
    for s in PROBE:
        print(f"  probe | {s}")
        for r in rows:
            if r["scenario"] == s:
                print(f"    {r['persona']:18s} {r['output']}")
    return rows


def train_epoch(order):
    model.train()
    losses = []
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    for n, batch in enumerate(batches, 1):
        loss = model(**collate(batch)).loss
        (loss / ACCUM).backward()
        losses.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            sched.step()
            opt.zero_grad()
    return sum(losses) / len(losses)


LOG = [{"epoch": 0, "train_loss": None, "held_out_loss": round(held_out_loss(), 4)}]
best = (LOG[0]["held_out_loss"], 0, lora_state())
PROBES, since_best = [], 0
rng = random.Random(SEED)
t0 = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_epoch(rng.sample(TRAIN, len(TRAIN)))
    LOG.append({"epoch": epoch, "train_loss": round(train_loss, 4), "held_out_loss": round(held_out_loss(), 4)})
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    PROBES += probe(epoch)
    if LOG[-1]["held_out_loss"] < best[0]:
        best, since_best = (LOG[-1]["held_out_loss"], epoch, lora_state()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower held-out loss for {PATIENCE} epochs")
        break

model.load_state_dict(best[2], strict=False)
BEST_EPOCH = best[1]
print(f"kept epoch {BEST_EPOCH} (held-out loss {best[0]}); epoch 0 = Qwen + Hinglish {LOG[0]['held_out_loss']}")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'epoch': 1, 'train_loss': 3.5929, 'held_out_loss': 3.4118} | 50s
  probe | Tumhara best friend apni shaadi ki khabar dene aaya hai.
    Vijay_Deewar       Aaj shaadi ke baar, tumhara friend ka khud ka khidmat nahi. Iska khud saath nahi, iska khidma nahi! Iska saath, is ka khad nahi hai. Is ka khuda nahi nahi ho. Is saath. Is khidra. Is khalisa. Is nahi sa
    Don                Aaj shaadi ke baar, tumhara friend ka khud ka khidmat nahi. Iska khud saath nahi hai. Isse nahi, iska khidma nahi hoga. Iske baad, tumhe apni saath ka khadma nikaalne ka khuda nahi hota. Ismein nahi n
    Shahenshah         Aaj shaadi ke baar, tumhara shaadi ka khud bhi dikhao. Iska khud, iska khuda. Isse tumhain saath saath. Lekin, tumhein saath nahi. Iske baad, tumne saath hai. Ismein saamne nahi, nahi saamane.
    Anthony_Gonsalves  Bhai, shaadi ke baar, tumhara saath nahi. Iska baar tumhamein nahi hain. Isse nahi, nahi toh. Iske

## Cell 5 — Generate: Qwen + Hinglish without and with the persona LoRA, same 84 inputs

Eight held-out and six new scenarios, six personas each. "Base" is the merged Qwen + Hinglish model with the persona LoRA switched off (`disable_adapter`), so the only difference is the persona LoRA. Each output records how it ended (stop token or token limit) and its Devanagari share.

In [20]:
# Cell 5 — Greedy generation for both variants

model.eval()
RESULTS = []
for variant in ["qwen-hi base", "qwen-hi + lora"]:
    t0 = time.perf_counter()
    for e in EVAL:
        if variant == "qwen-hi base":
            with model.disable_adapter():
                r = generate(e["input"])
        else:
            r = generate(e["input"])
        RESULTS.append({"variant": variant, **{k: e[k] for k in ("kind", "scenario", "persona", "reference")}, **r})
    print(f"{variant}: {len(EVAL)} outputs in {time.perf_counter() - t0:.0f}s")

qwen-hi base: 84 outputs in 270s
qwen-hi + lora: 84 outputs in 430s


## Cell 6 — Content quality: every output, side by side

For each scenario and persona: Qwen + Hinglish without the persona LoRA, then with it, then the reference line where one exists (held-out scenarios only). These are what the human rubric scores. A short summary of the automatic signals follows, for each variant and then for the LoRA model on held-out and new scenarios separately (generalization). They are diagnostic only, not quality scores.

In [21]:
# Cell 6 — Outputs per scenario and persona, then the automatic signals per variant

VARIANTS = list(dict.fromkeys(r["variant"] for r in RESULTS))


def get(variant, scenario, persona):
    return next(r for r in RESULTS if (r["variant"], r["scenario"], r["persona"]) == (variant, scenario, persona))


for s in HELD_OUT + NEW:
    print(f"\n=== [{'held-out' if s in HELD_OUT else 'new'}] {s}")
    for p in PERSONAS:
        print(f"  {p}")
        for v in VARIANTS:
            r = get(v, s, p)
            print(f"    {v:18s} | {r['output']}" + ("" if r["ended"] == "stop token" else "   [TOKEN LIMIT]"))
        if s in HELD_OUT:
            print(f"    {'reference':18s} | {get(VARIANTS[0], s, p)['reference']}")


def repeats_sentence(t):
    sentences = [s.strip().lower() for s in re.split(r"[.?!]", t) if len(s.strip()) > 12]
    return len(sentences) != len(set(sentences))


print("\nAUTOMATIC SIGNALS (diagnostic only)")
for v in VARIANTS:
    rs = [r for r in RESULTS if r["variant"] == v]
    n = len(rs)
    print(f"  {v:18s} | stop token {sum(r['ended'] == 'stop token' for r in rs)}/{n}"
          f" | repeats a sentence {sum(repeats_sentence(r['output']) for r in rs)}/{n}"
          f" | opens with 'Okay' {sum(r['output'].startswith('Okay') for r in rs)}/{n}"
          f" | any Devanagari {sum((r['devanagari_share'] or 0) > 0 for r in rs)}/{n}"
          f" | mostly Devanagari {sum((r['devanagari_share'] or 0) > 0.5 for r in rs)}/{n}"
          f" | median tokens {sorted(r['new_tokens'] for r in rs)[n // 2]}")
for kind in ("held-out", "new"):
    rs = [r for r in RESULTS if r["variant"] == "qwen-hi + lora" and r["kind"] == kind]
    n = len(rs)
    print(f"  lora, {kind:8s}     | stop token {sum(r['ended'] == 'stop token' for r in rs)}/{n}"
          f" | repeats a sentence {sum(repeats_sentence(r['output']) for r in rs)}/{n}"
          f" | median tokens {sorted(r['new_tokens'] for r in rs)[n // 2]}")


=== [held-out] Tumhara best friend apni shaadi ki khabar dene aaya hai.
  Vijay_Deewar
    qwen-hi base       | Vijay (Dekhna): "Bilkul, woh bhi aapke saath hai, lekin woh kuch bhi karta hai. Jab aapka paas koi kuch hai, toh woh usse bhi dikh sakti hai, aur woh apne bina kuch se bhi. Lekin   [TOKEN LIMIT]
    qwen-hi + lora     | Main toh bhi apni best friend ki shaadi khabari nahi hua. Main toh apni khushi aur apne family ko bhi. Lekin main apni baat karo, aur main bhi maine apni saath.⏤⏤⏤
⏤⏤ Maine apni khabo, aur apni family ko. Laga toh   [TOKEN LIMIT]
    reference          | Aaj tumhari aankhon mein jo khushi hai, uske liye tumne bahut intezaar kiya hai. Main har kadam par tumhare saath hoon.
  Don
    qwen-hi base       | Don: "Bilkul, woh bhi aapke best friend ke liye shaadi ke liya. Lekin woh kuch bhi karega, toh aapko bhi dhyan rakhna hoga. Aapko apne best friend ko shaadi mein apne saath se jaane ki koshish karna padega,   [TOKEN LIMIT]
    qwen-hi + lora     | Kya tumhari b

## Cell 7 — Persona differentiation (automatic)

For each scenario: the mean word overlap (Jaccard) between the six personas' lines, taken over all 15 pairs, and the number of identical lines. Lower overlap means the personas say different things. The same number on the dataset's own lines (all 28 scenarios) gives the yardstick for what the dataset itself calls *different*. Overlap is reported for held-out and new scenarios separately.

A second check for copying: the share of each output's word 4-grams that also appear in some training line. High values on new scenarios mean the model is reciting training lines instead of answering.

In [22]:
# Cell 7 — Overlap between personas per scenario, and 4-gram copying from the training lines

def words(t):
    return re.findall(r"[a-zऀ-ॿ]+", t.lower())


def jaccard(a, b):
    a, b = set(words(a)), set(words(b))
    return len(a & b) / len(a | b) if a | b else 1.0


def mean_overlap(lines):
    return sum(jaccard(a, b) for a, b in combinations(lines, 2)) / 15


def grams(t, n=4):
    w = words(t)
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}


TRAIN_GRAMS = set().union(*(grams(ex["target"]) for ex in TRAIN))


def copied(t):
    g = grams(t)
    return len(g & TRAIN_GRAMS) / len(g) if g else 0.0


ref = [mean_overlap([ex["target"] for ex in DATA if ex["scenario"] == s]) for s in SCENARIOS]
print(f"yardstick: dataset lines ({len(SCENARIOS)} scenarios), mean overlap between personas {sum(ref) / len(ref):.3f} "
      f"(range {min(ref):.3f}-{max(ref):.3f})\n")

DIFF = {}
print(f"{'scenario':58s} " + " | ".join(f"{v:>18s}" for v in VARIANTS))
for s in HELD_OUT + NEW:
    row = []
    for v in VARIANTS:
        lines = [get(v, s, p)["output"] for p in PERSONAS]
        DIFF[(v, s)] = {"overlap": round(mean_overlap(lines), 3), "identical": 6 - len(set(lines))}
        row.append(f"{DIFF[(v, s)]['overlap']:.3f} ({DIFF[(v, s)]['identical']} same)")
    print(f"{s[:58]:58s} " + " | ".join(f"{x:>18s}" for x in row))

def mean(xs):
    return sum(xs) / len(xs)


for v in VARIANTS:
    d = [DIFF[(v, s)]["overlap"] for s in HELD_OUT + NEW]
    print(f"\n{v:18s} | overlap: held-out {mean([DIFF[(v, s)]['overlap'] for s in HELD_OUT]):.3f}, "
          f"new {mean([DIFF[(v, s)]['overlap'] for s in NEW]):.3f}")
    c = [copied(r["output"]) for r in RESULTS if r["variant"] == v and r["kind"] == "new"]
    print(f"{v:18s} | mean overlap {sum(d) / len(d):.3f} | identical lines {sum(DIFF[(v, s)]['identical'] for s in HELD_OUT + NEW)}"
          f" | new scenarios: mean 4-gram copying {sum(c) / len(c):.2f}, outputs > 50% copied {sum(x > 0.5 for x in c)}/{len(c)}")

lora = [r for r in RESULTS if r["variant"] == "qwen-hi + lora"]
overlap = mean([DIFF[("qwen-hi + lora", s)]["overlap"] for s in HELD_OUT + NEW])
repeats = sum(repeats_sentence(r["output"]) for r in lora)
stops = sum(r["ended"] == "stop token" for r in lora)
print("\nSUCCESS BAR (automatic part; coherence and blind matching are human checks)")
print(f"  distinctiveness: overlap between personas {overlap:.3f} (pass <= 0.15, Gemma run 4 0.142): "
      f"{'PASS' if overlap <= 0.15 else 'FAIL'}")
print(f"  termination: stop token {stops}/{len(lora)} (pass >= 80): {'PASS' if stops >= 80 else 'FAIL'}")
print(f"  repetition: outputs repeating a sentence {repeats}/{len(lora)} (pass <= 4): {'PASS' if repeats <= 4 else 'FAIL'}")

yardstick: dataset lines (28 scenarios), mean overlap between personas 0.075 (range 0.037-0.130)

scenario                                                         qwen-hi base |     qwen-hi + lora
Tumhara best friend apni shaadi ki khabar dene aaya hai.       0.190 (0 same) |     0.269 (0 same)
Tumhari train chhoot gayi aur agli train kal subah hai.        0.206 (0 same) |     0.209 (0 same)
Ek purana dushman tumse madad maangne aaya hai.                0.242 (0 same) |     0.383 (0 same)
Doctor ne kaha hai ki tumhe meetha kam karna padega.           0.213 (0 same) |     0.280 (0 same)
Tumhari beti ne aaj pehli baar cycle chalayi.                  0.176 (0 same) |     0.267 (0 same)
Raat ko ghar ki bijli chali gayi aur sab dare hue hain.        0.267 (0 same) |     0.290 (0 same)
Jisne tumhara phone churaya tha, woh pakda gaya hai.           0.215 (0 same) |     0.207 (0 same)
Team ka naya member bahut sharmila hai aur meeting mein ku     0.205 (0 same) |     0.337 (0 same)
Boss ne Fri

## Cell 8 — Blind persona matching (for a human reviewer)

The real differentiation test. For each scenario, the LoRA model's six lines are shuffled and labelled A–F. A reviewer who knows the six characters matches each letter to a persona without looking at Cell 9. If the matches are mostly right, the personas are distinct and recognisable; if they are near chance (1 in 6), they are not.

The shuffle seed differs from every Gemma run, so no earlier key applies. The reviewer should be someone who has not seen this run's Cell 9.

In [23]:
# Cell 8 — Shuffled LoRA outputs per scenario, letters only (the key is in Cell 9)

KEY = {}
shuffle = random.Random(SEED + 5)                  # differs from every Gemma run
print("Personas: Vijay (Deewar), Don, Shahenshah, Anthony Gonsalves, Vijay Dinanath Chauhan (Agneepath), Jai (Sholay)")
for s in HELD_OUT + NEW:
    order = shuffle.sample(PERSONAS, len(PERSONAS))
    KEY[s] = dict(zip("ABCDEF", order))
    print(f"\n=== {s}")
    for letter, p in KEY[s].items():
        print(f"  {letter}. {get('qwen-hi + lora', s, p)['output']}")

Personas: Vijay (Deewar), Don, Shahenshah, Anthony Gonsalves, Vijay Dinanath Chauhan (Agneepath), Jai (Sholay)

=== Tumhara best friend apni shaadi ki khabar dene aaya hai.
  A. Aaj shaant rahe hain. Main apni best friend ke saath baat karenge. Isse aaram milegi.⏤⏤⏤
⏤⏤ Maine apni saath dekhaunga. Iska khaber main bhi dekhunga. Aaram milega.⏤
�
Aaram milegaa. Main bhi apni khushi
  B. Main toh bhi apni best friend ki shaadi khabari nahi hua. Main toh apni khushi aur apne family ko bhi. Lekin main apni baat karo, aur main bhi maine apni saath.⏤⏤⏤
⏤⏤ Maine apni khabo, aur apni family ko. Laga toh
  C. Bhai, main tumhari shaadi ka baat kahaunga. Main tumhari maa ki baat bhi kahaenge, lekin main tumne dekha. Aaj tumhari wife aur maa tumhari baat kar rahe hain. Aaram se baat hoga.⏤⏤⏤
⏤⏤ Maine tumhari bhi baat dekhte
  D. Bhai, main toh bhi koi koi baat nahi. Main apni best friend ke liye shaadi ke baar baat karna chahta hoon, lekin main nahi hoon. Aaj main apni baat karunga.⏤⏤⏤
⏤⏤ Maine apni

## Cell 9 — The key, and every record

The answer key for Cell 8, then the training log, settings, every result and every probe output as JSON lines between markers, so the scoring sheets can be rebuilt from the saved notebook alone.

In [24]:
# Cell 9 — Key for the blind sheet, settings, and the full records

print("KEY")
for s, k in KEY.items():
    print(f"  {s[:50]:50s} " + "  ".join(f"{letter}={p}" for letter, p in k.items()))

SETTINGS = {
    "versions": VERSIONS,
    "data": {"url": DATA_URL, "train": len(TRAIN), "held_out": len(HELD), "held_out_scenarios": HELD_OUT,
             "probe_scenarios": PROBE,
             "new_scenarios": NEW},
    "lora": {"r": LORA.r, "alpha": LORA.lora_alpha, "dropout": LORA.lora_dropout,
             "targets": LORA.target_modules},
    "decoder": {"base": BASE, "hinglish_adapter": ADAPTER, "revisions": REV, "hinglish_adapter_merged": True},
    "training": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "micro_batch": MICRO, "accumulation": ACCUM,
                 "lr": LR, "schedule": "linear decay over the maximum", "gradient_checkpointing": False,
                 "seed": SEED, "kept_epoch": BEST_EPOCH, "log": LOG},
    "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "stop_token_ids": STOPS,
                   "no_repeat_ngram_size": NO_REPEAT,
                   "template": "adapter chat template (plain ChatML), dataset input + traits line as the user turn, no system turn"},
    "traits": TRAITS,
}
print("\n----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for r in RESULTS:
    print(json.dumps(r, ensure_ascii=False))
for r in PROBES:
    print(json.dumps({"variant": "probe", **r}, ensure_ascii=False))
print("----- END RECORDS -----")

KEY
  Tumhara best friend apni shaadi ki khabar dene aay A=Shahenshah  B=Vijay_Deewar  C=Anthony_Gonsalves  D=Jai_Sholay  E=Don  F=Agneepath_Vijay
  Tumhari train chhoot gayi aur agli train kal subah A=Shahenshah  B=Agneepath_Vijay  C=Anthony_Gonsalves  D=Don  E=Vijay_Deewar  F=Jai_Sholay
  Ek purana dushman tumse madad maangne aaya hai.    A=Jai_Sholay  B=Vijay_Deewar  C=Agneepath_Vijay  D=Anthony_Gonsalves  E=Don  F=Shahenshah
  Doctor ne kaha hai ki tumhe meetha kam karna padeg A=Jai_Sholay  B=Agneepath_Vijay  C=Vijay_Deewar  D=Don  E=Shahenshah  F=Anthony_Gonsalves
  Tumhari beti ne aaj pehli baar cycle chalayi.      A=Don  B=Agneepath_Vijay  C=Shahenshah  D=Jai_Sholay  E=Vijay_Deewar  F=Anthony_Gonsalves
  Raat ko ghar ki bijli chali gayi aur sab dare hue  A=Agneepath_Vijay  B=Shahenshah  C=Jai_Sholay  D=Anthony_Gonsalves  E=Don  F=Vijay_Deewar
  Jisne tumhara phone churaya tha, woh pakda gaya ha A=Don  B=Vijay_Deewar  C=Jai_Sholay  D=Anthony_Gonsalves  E=Agneepath_Vijay  F=Shahen

TypeError: Object of type set is not JSON serializable

## Cell 10 — Optional: save the adapter

Writes the kept persona LoRA (about 75 MB; the merged Hinglish weights are not saved, they are rebuilt from the two public repositories) to `/kaggle/working/persona-pilot-qwen-adapter`. Files in `/kaggle/working` are lost when an interactive session ends, so download it if the pilot is worth keeping. Nothing is uploaded anywhere.

In [ ]:
# Cell 10 — Save the kept LoRA adapter locally on Kaggle

model.save_pretrained("/kaggle/working/persona-pilot-qwen-adapter")
!ls -la /kaggle/working/persona-pilot-qwen-adapter

---
## Next

Save the notebook. Then, side by side with Gemma 3 4B run 4 (same data, same checks):

1. **Stopping:** did the persona LoRA teach Qwen + Hinglish to stop (it never did on its own)?
2. **Coherence:** are at least 2 in 3 lines natural and on-topic, on held-out *and* new scenarios?
3. **Persona recognition:** a reviewer who has not seen Cell 9 fills in the blind sheet (Cell 8).
4. **Decision:** keep the decoder that passes more of the bar; if Qwen + Hinglish wins, it also matches the requirements' recommended decoder family.